# Exploratory Data Analysis

**Paper:** *Deep learning-based surrogate capacity models and multi-objective fragility estimates for reinforced concrete frames*
**Authors:** Lili Xing, Paolo Gardoni, Ge Song, Ying Zhou

## 1. Imports

In [5]:
import os
import numpy as np
import pandas as pd

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_DIR   = os.path.join(BASE_DIR, "1. Dataset", "DNN_dataset")
PROCESSED_DIR = os.path.join(BASE_DIR, "1. Dataset", "Python_DataProcess")

## 2. Inspect Raw Dataset Structure

In [6]:
# List the 50 simulation folders
folders = sorted(
    [d for d in os.listdir(DATASET_DIR) if d.startswith("RandomPushResults")],
    key=lambda x: int(x.replace("RandomPushResults", ""))
)
print(f"Folders: {len(folders)}  ({folders[0]} ... {folders[-1]})")

Folders: 80  (RandomPushResults1 ... RandomPushResults80)


In [ ]:
# Files inside each folder
folder1 = os.path.join(DATASET_DIR, "RandomPushResults1")
for f in sorted(os.listdir(folder1)):
    path = os.path.join(folder1, f)
    if "PushOver_Random_S" not in f and os.path.isfile(path):
        with open(path) as fh:
            n = sum(1 for l in fh if l.strip())
        print(f"{f}  ->  {n} rows")

In [ ]:
# InputVariablesR: 23 input features per scenario
iv = np.loadtxt(os.path.join(folder1, "PushOver_Random_InputVariablesR.txt"))
print("InputVariables shape:", iv.shape)

In [ ]:
# CapacityPointsR: 10 output columns per scenario
cp = np.loadtxt(os.path.join(folder1, "PushOver_Random_CapacityPointsR.txt"))
print("CapacityPoints shape:", cp.shape)
print()
print("Column mapping:")
for k, v in {
    0: "Dy  (mm)    -- top displacement at yield",
    1: "Vy  (kN)    -- base shear at yield  [top-disp curve]",
    2: "Du  (mm)    -- top displacement at peak",
    3: "Vu  (kN)    -- peak base shear  [= col 9]",
    4: "IDy_abs (mm)-- max inter-story drift at yield (absolute)",
    5: "IDu_abs (mm)-- max inter-story drift at peak  (absolute)",
    6: "IDy (frac)  -- max inter-story drift at yield (dimensionless)",
    7: "IVy (kN)    -- base shear at yield  [ISD curve]",
    8: "IDu (frac)  -- max inter-story drift at peak  (dimensionless)",
    9: "IVu (kN)    -- peak base shear  [ISD curve, = col 3]",
}.items():
    print(f"  col {k}: {v}")

In [ ]:
# Total scenarios across all 50 folders
total = 0
for folder in folders:
    with open(os.path.join(DATASET_DIR, folder, "PushOver_Random_availableS.txt")) as fh:
        total += sum(1 for l in fh if l.strip())
print(f"Total raw scenarios: {total:,}  ({total // len(folders)} per folder)")

## 3. Column Names

### 23 Sampled Input Features (Table 1 of the paper)

| Col | Symbol | Description | Unit |
|-----|--------|-------------|------|
| 0 | B | Breadth of one span | m |
| 1 | D | Depth of one span | m |
| 2 | H | Story height | m |
| 3 | B_num | Number of spans (X) | — |
| 4 | D_num | Number of spans (Y) | — |
| 5 | H_num | Number of storeys | — |
| 6 | colWidth | Column width (square) | m |
| 7 | beamDepth | Beam depth | m |
| 8 | beamRat | Beam depth-to-width ratio | — |
| 9 | Asc | Steel bar area in column | mm² |
| 10 | Asb | Steel bar area in beam | mm² |
| 11 | t | Slab thickness | m |
| 12 | cover1 | Concrete cover – column | m |
| 13 | cover2 | Concrete cover – beam | m |
| 14 | Ec | Concrete elastic tangent | ×10¹⁰ Pa |
| 15 | nu_c | Concrete Poisson ratio | — |
| 16 | fc | Concrete compressive strength | ×10⁶ Pa |
| 17 | fcuRat | Ratio fcu / fc | — |
| 18 | eps_cu | Concrete ultimate strain (sampled offset from eps_c0) | — |
| 19 | Es | Steel elastic tangent | ×10¹⁰ Pa |
| 20 | nu_s | Steel Poisson ratio | — |
| 21 | fsy | Steel yield strength | ×10⁶ Pa |
| 22 | eta | Strain-hardening ratio | — |

### 6 Derived / Fixed Parameters (Table 1, dash rows)

| Symbol | Formula / Value | Description |
|--------|----------------|-------------|
| eps_c0 | `2 * fc / Ec` (unit-adjusted) | Concrete strain at peak compressive strength |
| lambda_ | `0.1` (fixed) | Concrete02 unloading slope ratio |
| fct | `0.1 * fc` | Concrete tensile strength (×10⁶ Pa) |
| Ecs | `fct / 0.002` | Concrete tension softening stiffness |
| R0, cR1, cR2 | `20, 0.9215, 0.15` (fixed) | SteelMPF transition parameters |
| a1, a2, a3, a4 | `0, 1, 0, 0` (fixed) | SteelMPF isotropic hardening parameters |

### 8 Target Variables

| Col | Symbol | Description | Unit |
|-----|--------|-------------|------|
| 23 | Dy | Top displacement at yield | m |
| 24 | Vy | Base shear at yield | kN |
| 25 | Du | Top displacement at peak | m |
| 26 | Vu | Peak base shear | kN |
| 27 | IDy | Max inter-story drift at yield | % |
| 28 | IVy | Base shear at yield (ISD curve) | kN |
| 29 | IDu | Max inter-story drift at peak | % |
| 30 | IVu | Peak base shear (ISD curve) = Vu | kN |

## 4. Load Pre-Processed Dataset

`Python_DataProcess/` contains the final compiled files (31 cols = 23 inputs + 8 outputs):
- `RandomPushover_DL_Total_Train_new80.txt` — 80 % split
- `RandomPushover_DL_Total_Test_new80.txt`  — 20 % split

In [ ]:
INPUT_COLS = [
    "B", "D", "H", "B_num", "D_num", "H_num",
    "colWidth", "beamDepth", "beamRat",
    "Asc", "Asb", "t", "cover1", "cover2",
    "Ec", "nu_c", "fc", "fcuRat", "eps_cu",
    "Es", "nu_s", "fsy", "eta",
]
OUTPUT_COLS = ["Dy", "Vy", "Du", "Vu", "IDy", "IVy", "IDu", "IVu"]
ALL_COLS    = INPUT_COLS + OUTPUT_COLS

train_arr = np.loadtxt(os.path.join(PROCESSED_DIR, "RandomPushover_DL_Total_Train_new80.txt"))
test_arr  = np.loadtxt(os.path.join(PROCESSED_DIR, "RandomPushover_DL_Total_Test_new80.txt"))

print(f"Train : {train_arr.shape[0]:,} rows")
print(f"Test  : {test_arr.shape[0]:,} rows")
print(f"Total : {train_arr.shape[0] + test_arr.shape[0]:,} rows  x  {train_arr.shape[1]} cols")

## 5. Build the Clean DataFrame

In [ ]:
df_train = pd.DataFrame(train_arr, columns=ALL_COLS)
df_train["split"] = "train"

df_test = pd.DataFrame(test_arr, columns=ALL_COLS)
df_test["split"] = "test"

df = pd.concat([df_train, df_test], ignore_index=True)
print(f"Shape: {df.shape}")
df.head()

## 6. Add the 6 Derived / Fixed Parameters

These are not stored in the raw files but are required to fully define the OpenSees materials.
Computed directly from the sampled inputs using the formulas in Table 1.

In [ ]:
# eps_c0 = 2 * fc / Ec  (dimensionless)
# fc is in x1e6 Pa, Ec is in x1e10 Pa  =>  eps_c0 = 2 * fc*1e6 / (Ec*1e10) = 2*fc / (Ec*1e4)
df["eps_c0"]  = 2.0 * df["fc"] / (df["Ec"] * 1e4)

df["lambda_"] = 0.1          # fixed: Concrete02 unloading slope ratio

df["fct"]     = 0.1 * df["fc"]           # tensile strength (x1e6 Pa)
df["Ecs"]     = df["fct"] / 0.002        # tension softening stiffness

df["R0"]      = 20.0          # fixed: SteelMPF transition parameter
df["cR1"]     = 0.9215        # fixed
df["cR2"]     = 0.15          # fixed

df["a1"]      = 0.0           # fixed: SteelMPF isotropic hardening
df["a2"]      = 1.0
df["a3"]      = 0.0
df["a4"]      = 0.0

DERIVED_COLS = ["eps_c0", "lambda_", "fct", "Ecs", "R0", "cR1", "cR2", "a1", "a2", "a3", "a4"]

print(f"Shape after derived columns: {df.shape}")
df[["fc", "Ec", "eps_c0", "fct", "Ecs"]].head()

## 7. Quick Checks

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print()

# Vu and IVu must be identical (peak base shear is the same on both pushover curves)
assert np.allclose(df["Vu"].values, df["IVu"].values)
print("Vu == IVu for all rows: True")
print()

# IDu is capped at 2.4% (paper's prescribed inter-story drift limit)
print(f"IDu max: {df['IDu'].max():.4f}%  (paper limit: 2.4%)")

In [ ]:
print("=== Sampled Inputs ===")
df[INPUT_COLS].describe().T.round(4)

In [ ]:
print("=== Derived Parameters ===")
df[DERIVED_COLS].describe().T.round(6)

In [ ]:
print("=== Target Variables ===")
df[OUTPUT_COLS].describe().T.round(4)

## 8. Export to CSV

In [ ]:
output_csv = os.path.join(BASE_DIR, "1. Dataset", "DNN_dataset_cleaned.csv")
df.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")
print(f"Shape: {df.shape}  ({os.path.getsize(output_csv)/1024**2:.1f} MB)")

In [ ]:
# Verify reload
df_check = pd.read_csv(output_csv)
print(f"Reloaded: {df_check.shape}")
print(f"Columns : {list(df_check.columns)}") 